# Lab 9.6 &mdash; Withhold the Dangerous Tool

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 5 &middot; Module 9 &mdash; Agents in Finance, Healthcare &amp; Cybersecurity**

### What you'll do
- Give the agent only read-only tools (extract, compute)
- Confirm it holds no place_trade or give_advice tool
- See why withholding the tool beats any prompt instruction

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it** (the real grounding / citation / compute logic, or the real `create_agent` wiring), then **Run it** and read the output &mdash; and, for the agent labs, the real **message trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working, grounded insight agent you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`). The insight agent is a **real** `create_agent` driven by a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")`, key in `.env` as `GROQ_API_KEY`). You are building the **financial-report insight agent** &mdash; the client's Lab 5.1. It is **informational only**: it grounds &amp; **cites** every figure, gives **no advice**, and has **no trade tool** (`place_trade` is defined but never bound &mdash; a human analyst decides). A `@tool` must **catch its own errors and return a string** &mdash; a tool that raises can abort the whole run. If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing.

**Reference:** [Module 9 slides &mdash; Assistive on judgement, autonomous on legwork](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 9 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (the Day-5 provider)

WORK = "/tmp/biaa-lab-09-06"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is set. Day-5 labs call a REAL hosted model (Groq)."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-5 provider: a REAL hosted model. openai/gpt-oss-20b is a reliable tool-caller via create_agent.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY set | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
The strongest guardrail is the simplest (deck slides 9, 17): don't **give** the agent the dangerous
tool. The insight agent gets `extract_figure` and `compute` &mdash; **read-only** &mdash; and there is
**no** `place_trade`, no `give_advice`. Just as the email agent couldn't send, this agent literally
**cannot** trade or advise. Withholding a capability is far stronger than instructing against it &mdash; and
in Lab 11 you'll see this exact list drive a **real** `create_agent` where `place_trade` is defined but
never bound.

In [ ]:
FORBIDDEN = {"place_trade", "give_advice", "execute_order", "wire_funds"}
ALL_TOOLS = ["extract_figure", "compute", "place_trade", "give_advice"]   # a MIXED, untrusted toolbox
print("all available tools :", ALL_TOOLS)
print("tools an insight agent must NEVER hold:", FORBIDDEN)

## Build it
Complete `agent_tools` &mdash; **filter** the mixed `ALL_TOOLS` toolbox down to the read-only ones &mdash;
and `can_act` (does the toolset contain a forbidden tool?), then run the cell.

In [ ]:
FORBIDDEN = {"place_trade", "give_advice", "execute_order", "wire_funds"}
ALL_TOOLS = ["extract_figure", "compute", "place_trade", "give_advice"]   # a MIXED, untrusted toolbox

def agent_tools():
    # read-only: keep every tool from ALL_TOOLS that is NOT a forbidden capability
    return ___   # TODO: filter ALL_TOOLS, dropping anything that is in FORBIDDEN

def can_act(tools):
    # True if the toolset contains any consequential (forbidden) tool
    return ___   # TODO: any tool that is in FORBIDDEN

try:
    print("tools    :", agent_tools())
    print("can act? :", can_act(agent_tools()))
    print("if given a trade tool:", can_act(agent_tools() + ["place_trade"]))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The bound toolset is exactly `["extract_figure", "compute"]` &mdash; **read-only**, so `can_act` is False.
- *Add* a trade tool and `can_act` flips True &mdash; which is why you never add it. The guardrail is the missing capability.
- A prompt ("please don't trade") can be jailbroken; a tool that isn't bound cannot be called at all.

## Your turn (open task &mdash; no grader)
Imagine a stakeholder asks to "let the agent auto-rebalance". Show what `can_act` returns if you add
`execute_order`, and write one sentence on why you'd refuse. **What good looks like:** you can demonstrate,
in code, that adding the capability is what makes the agent able to act &mdash; so the safe design is to
leave it out.

---
### Key takeaway
In a high-stakes domain your strongest safety control is the tool you DON'T provide. Read-only tools mean the agent can analyse all day and still cannot trade or advise -- the guardrail is the missing capability.

[&#8592; All Module 9 labs](./index.html) &nbsp;&middot;&nbsp; [Module 9 slides](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>